# opencensus_context smoke test

The `opencensus-context` dist ships exactly one module: `opencensus.common.runtime_context`,
exposing the `RuntimeContext` singleton (contextvars-backed on modern Pythons). There is no
`opencensus.context` module — the examples below exercise the real API surface: slot
registration, attribute get/set, snapshot/apply, cross-thread propagation, and clear().

In [ ]:
# Submodule smoke test: import the package + walk every submodule.
import pkgutil, importlib
import opencensus.common.runtime_context as _pkg_under_test  # the module the opencensus-context dist ships

failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix=f"{_pkg_under_test.__name__}."):
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failed.append((module_info.name, type(e).__name__, str(e)))

if failed:
    for name, etype, msg in failed:
        print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of {_pkg_under_test.__name__} failed to import")
print(f"OK — all submodules of {_pkg_under_test.__name__} imported cleanly")

In [ ]:
# Example 1: Import the RuntimeContext singleton and show which implementation is active.
import importlib.metadata

from opencensus.common.runtime_context import RuntimeContext

print(f"dist version: {importlib.metadata.version('opencensus-context')}")
print(f"implementation: {type(RuntimeContext).__name__}")
print(f"repr: {RuntimeContext!r}")

# contextvars-backed on any modern Python; thread-local fallback otherwise.
assert type(RuntimeContext).__name__ in ("_AsyncRuntimeContext", "_ThreadLocalRuntimeContext")

In [ ]:
# Example 2: register_slot + attribute-style set/get (the primary operation).
from opencensus.common.runtime_context import RuntimeContext

slot = RuntimeContext.register_slot("ex2_trace_id", default="unset")
print(f"registered slot: {slot.name}")
print(f"default value: {RuntimeContext.ex2_trace_id}")
assert RuntimeContext.ex2_trace_id == "unset"

RuntimeContext.ex2_trace_id = "abc123"
print(f"after set: {RuntimeContext.ex2_trace_id}")
assert RuntimeContext.ex2_trace_id == "abc123"

# Callable defaults are evaluated lazily on first read.
RuntimeContext.register_slot("ex2_counter", default=lambda: 0)
assert RuntimeContext.ex2_counter == 0
print("callable default OK")

In [ ]:
# Example 3: snapshot() + apply() round-trip restores earlier context state.
from opencensus.common.runtime_context import RuntimeContext

RuntimeContext.register_slot("ex3_user", default=None)
RuntimeContext.ex3_user = "alice"

snap = RuntimeContext.snapshot()
print(f"snapshot captured: ex3_user={snap['ex3_user']}")
assert snap["ex3_user"] == "alice"

RuntimeContext.ex3_user = "bob"
assert RuntimeContext.ex3_user == "bob"

RuntimeContext.apply(snap)  # restore the earlier state
assert RuntimeContext.ex3_user == "alice"
print("snapshot/apply round-trip OK")

In [ ]:
# Example 4: with_current_context propagates the caller's context into another thread.
import threading

from opencensus.common.runtime_context import RuntimeContext

RuntimeContext.register_slot("ex4_request_id", default="none")
RuntimeContext.ex4_request_id = "req-42"

seen = {}

def handler():
    seen["request_id"] = RuntimeContext.ex4_request_id

# A fresh thread would only see the slot default; the wrapper carries the caller's context.
t = threading.Thread(target=RuntimeContext.with_current_context(handler))
t.start()
t.join()

print(f"propagated into thread: {seen['request_id']}")
assert seen["request_id"] == "req-42"

In [ ]:
# Example 5: clear() resets slots to defaults; misuse fails loudly as documented.
from opencensus.common.runtime_context import RuntimeContext

RuntimeContext.register_slot("ex5_flag", default=False)
RuntimeContext.ex5_flag = True
assert RuntimeContext.ex5_flag is True

RuntimeContext.clear()  # resets every registered slot to its default
assert RuntimeContext.ex5_flag is False
print("clear() reset slots to defaults")

# Reading an unregistered slot raises AttributeError (documented behavior).
try:
    RuntimeContext.not_a_slot
    raise AssertionError("expected AttributeError for unregistered slot")
except AttributeError as error:
    print(f"AttributeError as documented: {error}")

# Duplicate registration is rejected with ValueError (documented behavior).
try:
    RuntimeContext.register_slot("ex5_flag")
    raise AssertionError("expected ValueError for duplicate slot")
except ValueError as error:
    print(f"ValueError as documented: {error}")